Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Load model and reevaluate on clean and master dataset

In [2]:
import onnx
from onnx2pytorch import ConvertModel

onnx_model = onnx.load("baseline_resnet152v2.onnx")
torch_model = ConvertModel(onnx_model)
torch_model.eval()

ConvertModel(
  (Transpose_functional_1/resnet152v2_1/conv1_conv_1/convolution__6:0): Transpose()
  (Conv_Conv__643:0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
  (Pad_functional_1/resnet152v2_1/pool1_pad_1/Pad:0): Pad(mode=constant, padding=None)
  (MaxPool_functional_1/resnet152v2_1/pool1_pool_1/MaxPool2d:0): MaxPool2d(kernel_size=(3, 3), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (Mul_functional_1/resnet152v2_1/conv2_block1_preact_bn_1/batchnorm/mul_1:0): mul()
  (Add_functional_1/resnet152v2_1/conv2_block1_preact_bn_1/batchnorm/add_1:0): Add()
  (Relu_functional_1/resnet152v2_1/conv2_block1_preact_relu_1/Relu:0): ReLU(inplace=True)
  (Conv_functional_1/resnet152v2_1/conv2_block1_0_conv_1/BiasAdd:0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
  (Conv_Conv__651:0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
  (Relu_functional_1/resnet152v2_1/conv2_block1_1_relu_1/Relu:0): ReLU(inplace=True)
  (Conv_Conv__653:0): Conv2d(64, 64, kern

FGSM Attack and Evaluate Functions

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()

def fgsm_attack(model, images, labels, epsilon=0.01):
    """Generate FGSM adversarial examples."""
    images = images.clone().detach().to(device).float()
    labels = labels.clone().detach().to(device).float().view(-1, 1)

    images.requires_grad = True

    predictions = model(images)
    loss = loss_fn(predictions, labels)
    model.zero_grad()
    loss.backward()

    adv_images = images + epsilon * images.grad.sign()
    adv_images = torch.clamp(adv_images, 0, 1).detach()

    return adv_images

In [ ]:
def evaluate_clean(model, loader):
    """Evaluate model on clean (unperturbed) data."""
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device).float()
            labels = labels.to(device).float()

            logits = model(images).view(-1)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_labels.extend(labels.cpu().numpy().astype(int))
            all_probs.extend(probs)
            all_preds.extend(preds)

    acc = accuracy_score(all_labels, all_preds)
    print(f"Clean Accuracy: {acc:.4f}")
    return np.array(all_labels), np.array(all_preds), np.array(all_probs), acc

In [ ]:
def evaluate_fgsm(model, loader, epsilon=0.01):
    """Evaluate model on FGSM adversarial examples."""
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    for images, labels in loader:
        images = images.to(device).float()
        labels = labels.to(device).float()

        adv_images = fgsm_attack(model, images, labels, epsilon=epsilon)

        with torch.no_grad():
            logits = model(adv_images).view(-1)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int))
        all_probs.extend(probs)
        all_preds.extend(preds)

    acc = accuracy_score(all_labels, all_preds)
    print(f"FGSM Accuracy (eps={epsilon}): {acc:.4f}")
    return np.array(all_labels), np.array(all_preds), np.array(all_probs), acc

Evaluate on their clean dataset

In [ ]:
IMG_SIZE = 224
BATCH = 16

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),  # also scales to [0, 1]
])

Clean dataset (theirs)

In [ ]:
ds_clean = datasets.ImageFolder(root="chest_xray/test", transform=transform)
loader_clean = DataLoader(ds_clean, batch_size=BATCH, shuffle=False)

print("Clean classes:", ds_clean.class_to_idx)

Master Dataset

In [ ]:
ds_master = datasets.ImageFolder(root="Master_Dataset/test", transform=transform)
loader_master = DataLoader(ds_master, batch_size=BATCH, shuffle=False)

print("Master classes:", ds_master.class_to_idx)

Kermany Dataset

In [ ]:
ds_kermany = datasets.ImageFolder(root="Kermany_Pediatric_Attack", transform=transform)
loader_kermany = DataLoader(ds_kermany, batch_size=BATCH, shuffle=False)

print("Kermany classes:", ds_kermany.class_to_idx)

Evaluate both theirs and master

In [ ]:
labels_clean, preds_clean, _, clean_acc = evaluate_clean(model, loader_clean)
labels_master, preds_master, _, master_acc = evaluate_clean(model, loader_master)
labels_kermany, preds_kermany, _, kermany_acc = evaluate_clean(model, loader_kermany)

print("Clean (chest_xray):", clean_acc)
print("Clean (master):", master_acc)
print("Clean (kermany):", kermany_acc)

Evaluate on FGSM on both

In [ ]:
eps = 0.01

_, _, _, fgsm_clean_acc = evaluate_fgsm(model, loader_clean, epsilon=eps)
_, _, _, fgsm_master_acc = evaluate_fgsm(model, loader_master, epsilon=eps)
_, _, _, fgsm_kermany_acc = evaluate_fgsm(model, loader_kermany, epsilon=eps)

print("FGSM (chest_xray):", fgsm_clean_acc)
print("FGSM (master):", fgsm_master_acc)
print("FGSM (kermany):", fgsm_kermany_acc)

Adversarial Training

In [ ]:
def adversarial_train_step(model, optimizer, images, labels, epsilon=0.01):
    """Single adversarial training step: trains on clean + FGSM examples."""
    model.train()
    images = images.to(device).float()
    labels = labels.to(device).float().view(-1, 1)

    adv_images = fgsm_attack(model, images, labels, epsilon=epsilon)

    combined_images = torch.cat([images, adv_images], dim=0)
    combined_labels = torch.cat([labels, labels], dim=0)

    model.train()
    optimizer.zero_grad()
    predictions = model(combined_images)
    loss = loss_fn(predictions, combined_labels)
    loss.backward()
    optimizer.step()

    return loss.item()

In [ ]:
def adversarial_train(model, train_loader, val_loader, epochs=5, epsilon=0.01, lr=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        model.train()
        train_losses = []

        for images, labels in train_loader:
            loss = adversarial_train_step(model, optimizer, images, labels, epsilon=epsilon)
            train_losses.append(loss)

        avg_train_loss = np.mean(train_losses)

        # Validation
        model.eval()
        val_losses = []
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device).float()
                labels = labels.to(device).float().view(-1, 1)
                logits = model(images)
                v_loss = loss_fn(logits, labels)
                val_losses.append(v_loss.item())
                preds = (torch.sigmoid(logits) > 0.5).int().cpu().numpy().flatten()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy().astype(int).flatten())

        val_acc = accuracy_score(all_labels, all_preds)
        avg_val_loss = np.mean(val_losses)
        print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

Datasets for adversarial training

In [ ]:
IMG_SIZE = 224
BATCH = 16
SEED = 42

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

full_train_ds = datasets.ImageFolder(root="chest_xray/train", transform=train_transform)

val_size = int(0.2 * len(full_train_ds))
train_size = len(full_train_ds) - val_size
ds_train, ds_val = random_split(
    full_train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

loader_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True)
loader_val = DataLoader(ds_val, batch_size=BATCH, shuffle=False)

print(f"Train size: {len(ds_train)} | Val size: {len(ds_val)}")

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model("baseline_resnet152v2.keras")

adversarial_train(
    model,
    loader_train,
    loader_val,
    epochs=3,
    epsilon=0.01,
    lr=1e-5
)

In [ ]:
torch.save(model.state_dict(), "baseline_resnet152v2_adversarial.pth")
print("Model saved to baseline_resnet152v2_adversarial.pth")

In [ ]:
# Use loader_clean as the test split here; replace with a dedicated test loader if available
labels_test, preds_test, probs_test, clean_test_acc = evaluate_clean(model, loader_clean)
print("Test Clean Accuracy:", clean_test_acc)

In [ ]:
labels_test, preds_test, probs_test, fgsm_test_acc = evaluate_fgsm(model, loader_clean)
print("Test FGSM Accuracy:", fgsm_test_acc)

==================================================================

Adversarial Training on Master

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model("baseline_resnet152v2_adversarial.keras")
model.summary()

Dataset for Master

In [ ]:
IMG_SIZE = 224
BATCH = 16
SEED = 42

master_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

ds_master_train_full = datasets.ImageFolder(root="Master_Dataset/train", transform=master_transform)

master_val_size = int(0.2 * len(ds_master_train_full))
master_train_size = len(ds_master_train_full) - master_val_size
ds_master_train_split, _ = random_split(
    ds_master_train_full, [master_train_size, master_val_size],
    generator=torch.Generator().manual_seed(SEED)
)

ds_master_val_ds = datasets.ImageFolder(root="Master_Dataset/val", transform=master_transform)

loader_master_train = DataLoader(ds_master_train_split, batch_size=BATCH, shuffle=True)
loader_master_val = DataLoader(ds_master_val_ds, batch_size=BATCH, shuffle=False)

ds_master_test = datasets.ImageFolder(root="Master_Dataset/test", transform=master_transform)
loader_master_test = DataLoader(ds_master_test, batch_size=BATCH, shuffle=False)

print(f"Master Train: {len(ds_master_train_split)} | Val: {len(ds_master_val_ds)} | Test: {len(ds_master_test)}")

Regular Training Functions

In [ ]:
def train_step(model, optimizer, images, labels):
    model.train()
    images = images.to(device).float()
    labels = labels.to(device).float().view(-1, 1)

    optimizer.zero_grad()
    predictions = model(images)
    loss = loss_fn(predictions, labels)
    loss.backward()
    optimizer.step()

    return loss.item()


def train(model, train_loader, val_loader, epochs=5, lr=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        model.train()
        train_losses = []

        for images, labels in train_loader:
            loss = train_step(model, optimizer, images, labels)
            train_losses.append(loss)

        avg_train_loss = np.mean(train_losses)

        # Validation
        model.eval()
        val_losses = []
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device).float()
                labels = labels.to(device).float().view(-1, 1)
                logits = model(images)
                v_loss = loss_fn(logits, labels)
                val_losses.append(v_loss.item())
                preds = (torch.sigmoid(logits) > 0.5).int().cpu().numpy().flatten()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy().astype(int).flatten())

        val_acc = accuracy_score(all_labels, all_preds)
        avg_val_loss = np.mean(val_losses)
        print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

Training

In [ ]:
train(
    model,
    loader_master_train,
    loader_master_val,
    epochs=15,
    lr=1e-5
)

In [ ]:
labels_master_test, preds_master_test, _, master_test_acc = evaluate_clean(model, loader_master_test)
print("Test Accuracy:", master_test_acc)

In [ ]:
torch.save(model.state_dict(), "baseline_resnet152v2_adversarial_finetunedonmaster.pth")
print("Model saved to baseline_resnet152v2_adversarial_finetunedonmaster.pth")